# NB29 — HRNet result audit and report
**CPU · ONE copy · Internet ON · HF_TOKEN · attach nothing · Run All.**
Run after NB28 completes all three seeds. Verifies epoch-60 statuses, test-image
coverage and recalculates point errors. Reports per-tyre/seed metrics and a simple
constant-coordinate baseline fitted only on training labels. Downloads no weights.
This is NOT a matched HRNet-versus-SegFormer result or proof of full S9 completion.
Publishes one small result report to HF. Send executed notebooks for verification.

User confirms 12 sessions are 12 different tyres. Identity source is recorded as
user-confirmed, not independently verified. No new labels requested. All 720
submitted points are visible: this is coordinate-only HRNet-W18, not learned
visibility, healthy-reference detection, or physical-angle estimation.
Dataset images come from attached Kaggle input; annotation JSON is pinned on HF.
No four-worker execution: use ONE copy. Existing notebooks/labels are untouched.


In [1]:
import sys, subprocess, importlib.util, base64, json, os
from pathlib import Path
WORK=Path('/kaggle/working/hrnet_geometry');WORK.mkdir(exist_ok=True)
if importlib.util.find_spec('huggingface_hub') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.36,<2'])
(WORK/'hrnet_protocol.py').write_bytes(base64.b64decode('IiIiRnJvemVuLCB1c2VyLWlkZW50aXR5LWNvbmZpcm1lZCBIUk5ldCBnZW9tZXRyeSBleHBlcmltZW50LiBDUFUgcHJlZmxpZ2h0LiIiIgppbXBvcnQgY29sbGVjdGlvbnMKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgdGltZQppbXBvcnQgcmVxdWVzdHMKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZURyYXcKClJFUE89J1NoYW5tdWs0NjIyL3R5cmUtd2Vhci1zdHVkeScKUkVWPScyYjQ3NzM5MTRlNmVlZmU0MjFkOGRkZDc0ZTc5YzViODQ2MDZjOWFhJwpQQUNLQUdFPScxMDBkYWM1YjEyZTExOTJlZGU1OWY5ZDEyMzFjNWFlZWVkODA0Mjg4NTA2NjFhOWFkZDFjYmExNDdmZTIyZjkyJwpBTk49JzhlOGZkMDczNGI5ZjRmZTQyMzZlNjk5YjgwNGMyZjhhZmI4NWFlNGJiODQ3ZjhkM2M4M2IwYWJmZjVhODc2OWUnCkJBU0U9ZidzOS9zOS1nZW9tZXRyeS1sYWJlbHMtcjItYWxsMTIwL3BhY2thZ2VzL3tQQUNLQUdFfScKVkVSU0lPTj0naHJuZXQtZ2VvbWV0cnktMjAyNi0wOS0xNS1yMScKUE9JTlRTPVsnbGVmdF91cHBlcicsJ3JpZ2h0X3VwcGVyJywnbGVmdF9taWRkbGUnLCdyaWdodF9taWRkbGUnLCdsZWZ0X2xvd2VyJywncmlnaHRfbG93ZXInXQpkZWYgY2Fub25pY2FsKHgpOnJldHVybiBqc29uLmR1bXBzKHgsc29ydF9rZXlzPVRydWUsc2VwYXJhdG9ycz0oJywnLCc6JyksZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoKQpkZWYgc2hhKHgpOnJldHVybiBoYXNobGliLnNoYTI1Nih4KS5oZXhkaWdlc3QoKQpkZWYgZmlsZV9zaGEocGF0aCk6CiAgICBoPWhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCdyYicpIGFzIGY6CiAgICAgICAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDEwMjQqKjIpLGInJyk6aC51cGRhdGUoYikKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCmRlZiB3cml0ZShwYXRoLHgpOgogICAgcGF0aD1QYXRoKHBhdGgpO3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgdG1wPXBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXgrJy50bXAnKTt0bXAud3JpdGVfYnl0ZXMoY2Fub25pY2FsKHgpKTt0bXAucmVwbGFjZShwYXRoKQpkZWYgcmVhZF9qc29uKHBhdGgpOnJldHVybiBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQpkZWYgcmV0cnkoZm4pOgogICAgZm9yIGkgaW4gcmFuZ2UoNik6CiAgICAgICAgdHJ5OnJldHVybiBmbigpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICByZXNwb25zZT1nZXRhdHRyKGUsJ3Jlc3BvbnNlJyxOb25lKTtjb2RlPWdldGF0dHIocmVzcG9uc2UsJ3N0YXR1c19jb2RlJyxOb25lKQogICAgICAgICAgICBpZiBjb2RlIG5vdCBpbiAoTm9uZSw0MjksNTAwLDUwMiw1MDMsNTA0KSBvciBpPT01OnJhaXNlCiAgICAgICAgICAgIGhpbnQ9Z2V0YXR0cihyZXNwb25zZSwnaGVhZGVycycse30pLmdldCgnUmV0cnktQWZ0ZXInLCcwJykKICAgICAgICAgICAgZGVsYXk9bWF4KDEwKjIqKmksZmxvYXQoaGludCkgaWYgc3RyKGhpbnQpLmlzZGlnaXQoKSBlbHNlIDApCiAgICAgICAgICAgIHByaW50KGYnSEYgcmV0cnkgYWZ0ZXIge2RlbGF5Oi4wZn1zOyBkdXJhYmxlIGxvY2FsIHN0YXRlIHJldGFpbmVkLicsZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgdW50aWw9dGltZS5tb25vdG9uaWMoKStkZWxheQogICAgICAgICAgICB3aGlsZSB0aW1lLm1vbm90b25pYygpPHVudGlsOnRpbWUuc2xlZXAobWF4KDAsbWluKDUsdW50aWwtdGltZS5tb25vdG9uaWMoKSkpKQpkZWYgc291cmNlKHBhdGgsd29yayk6CiAgICBkZXN0PVBhdGgod29yaykvJ3NvdXJjZXMnL3NoYShwYXRoLmVuY29kZSgpKTtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIG5vdCBkZXN0LmV4aXN0cygpOgogICAgICAgIGRlZiBkb3dubG9hZCgpOgogICAgICAgICAgICB3aXRoIHJlcXVlc3RzLmdldChmJ2h0dHBzOi8vaHVnZ2luZ2ZhY2UuY28vZGF0YXNldHMve1JFUE99L3Jlc29sdmUve1JFVn0ve0JBU0V9L3twYXRofScsdGltZW91dD00NSxzdHJlYW09VHJ1ZSkgYXMgcjoKICAgICAgICAgICAgICAgIHIucmFpc2VfZm9yX3N0YXR1cygpO2I9Ynl0ZWFycmF5KCkKICAgICAgICAgICAgICAgIGZvciBjaHVuayBpbiByLml0ZXJfY29udGVudCg2NTUzNik6CiAgICAgICAgICAgICAgICAgICAgYi5leHRlbmQoY2h1bmspCiAgICAgICAgICAgICAgICAgICAgaWYgbGVuKGIpPjIqMTAyNCoqMjpyYWlzZSBWYWx1ZUVycm9yKCdNZXRhZGF0YSBjYXAgZXhjZWVkZWQnKQogICAgICAgICAgICByZXR1cm4gYgogICAgICAgIHJhdz1yZXRyeShkb3dubG9hZCk7dG1wPWRlc3Qud2l0aF9zdWZmaXgoJy50bXAnKTt0bXAud3JpdGVfYnl0ZXMocmF3KTt0bXAucmVwbGFjZShkZXN0KQogICAgcmV0dXJuIGRlc3QucmVhZF9ieXRlcygpCmRlZiBzcGxpdF9ncm91cHMoaW1hZ2VzKToKICAgICIiIjgvMi8yIHR5cmVzOyBjaG9vc2UgaW1hZ2UtY291bnQgYmFsYW5jZSwgdGhlbiBhIGZpeGVkIGhhc2ggdGllLWJyZWFrIG9ubHkuIiIiCiAgICBjb3VudHM9Y29sbGVjdGlvbnMuQ291bnRlcihyWydzZXNzaW9uJ10gZm9yIHIgaW4gaW1hZ2VzKTtncm91cHM9c29ydGVkKGNvdW50cykKICAgIGFzc2VydCBsZW4oZ3JvdXBzKT09MTIKICAgIG9wdGlvbnM9W10KICAgIGZvciB0ZXN0IGluIGl0ZXJ0b29scy5jb21iaW5hdGlvbnMoZ3JvdXBzLDIpOgogICAgICAgIGZvciBkZXYgaW4gaXRlcnRvb2xzLmNvbWJpbmF0aW9ucyhbZyBmb3IgZyBpbiBncm91cHMgaWYgZyBub3QgaW4gdGVzdF0sMik6CiAgICAgICAgICAgIG50PXN1bShjb3VudHNbZ10gZm9yIGcgaW4gdGVzdCk7bnY9c3VtKGNvdW50c1tnXSBmb3IgZyBpbiBkZXYpCiAgICAgICAgICAgIGlmIG1pbihudCxudik8MTI6Y29udGludWUKICAgICAgICAgICAga2V5PShhYnMobnQtMjQpK2Ficyhudi0yNCksc2hhKGNhbm9uaWNhbChbdGVzdCxkZXZdKSkpCiAgICAgICAgICAgIG9wdGlvbnMuYXBwZW5kKChrZXksbGlzdCh0ZXN0KSxsaXN0KGRldikpKQogICAgXyx0ZXN0LGRldj1taW4ob3B0aW9ucykKICAgIHJldHVybiBkaWN0KHRyYWluPVtnIGZvciBnIGluIGdyb3VwcyBpZiBnIG5vdCBpbiB0ZXN0K2Rldl0sdmFsaWRhdGlvbj1kZXYsdGVzdD10ZXN0KQpkZWYgY29udHJhY3Qod29yayk6CiAgICByYXc9c291cmNlKGYncmV2aWV3cy97QU5OfS9BTk5PVEFUSU9OUy5qc29uJyx3b3JrKTthc3NlcnQgc2hhKHJhdyk9PUFOTgogICAgbGFiZWxzPWpzb24ubG9hZHMocmF3KTttYW5pZmVzdD1qc29uLmxvYWRzKHNvdXJjZSgnTUFOSUZFU1QuanNvbicsd29yaykpCiAgICBhc3NlcnQgc2hhKGNhbm9uaWNhbCh7azp2IGZvciBrLHYgaW4gbWFuaWZlc3QuaXRlbXMoKSBpZiBrIT0ncGFja2FnZV9pZCd9KSk9PVBBQ0tBR0UKICAgIGltcG9ydCBzOV9leHBhbnNpb24gYXMgZQogICAgY2hlY2tzPWUudmFsaWRhdGUobGFiZWxzLG1hbmlmZXN0KQogICAgYXNzZXJ0IGxlbihjaGVja3MpPT0xMjAgYW5kIGFsbChyWydjb21wbGV0ZSddIGZvciByIGluIGNoZWNrcykKICAgIGJ5X2lkPXtyWydwaWxvdF9pZCddOnIgZm9yIHIgaW4gbGFiZWxzWydhbm5vdGF0aW9ucyddfQogICAgYXNzZXJ0IGFsbChieV9pZFtyWydwaWxvdF9pZCddXVsncG9pbnRzJ11bbl1bJ3N0YXRlJ109PSd2aXNpYmxlJyBmb3IgciBpbiBtYW5pZmVzdFsnaW1hZ2VzJ10gZm9yIG4gaW4gUE9JTlRTKSwgJ1RoaXMgcHJvdG9jb2wgaXMgY29vcmRpbmF0ZS1vbmx5IGZvciB0aGUgZnJvemVuIGFsbC12aXNpYmxlIHN1Ym1pc3Npb24nCiAgICBncm91cHM9c3BsaXRfZ3JvdXBzKG1hbmlmZXN0WydpbWFnZXMnXSkKICAgIHJvd3M9W10KICAgIGZvciByZWYgaW4gbWFuaWZlc3RbJ2ltYWdlcyddOgogICAgICAgIGE9YnlfaWRbcmVmWydwaWxvdF9pZCddXQogICAgICAgIHJvbGU9bmV4dChrIGZvciBrLHYgaW4gZ3JvdXBzLml0ZW1zKCkgaWYgcmVmWydzZXNzaW9uJ10gaW4gdikKICAgICAgICByb3dzLmFwcGVuZChyZWZ8ZGljdChyb2xlPXJvbGUseD1bYVsncG9pbnRzJ11bbl1bJ3gnXS8ocmVmWyd3aWR0aCddLTEpIGZvciBuIGluIFBPSU5UU10pKQogICAgY2ZnPWRpY3QodmVyc2lvbj1WRVJTSU9OLHNvdXJjZV9yZXZpc2lvbj1SRVYsYW5ub3RhdGlvbnNfc2hhMjU2PUFOTixwYWNrYWdlX2lkPVBBQ0tBR0UsCiAgICAgICAgaWRlbnRpdHk9ZGljdChzb3VyY2U9J1VzZXIgY29uZmlybWF0aW9uIGluIHByb2plY3QgY29udmVyc2F0aW9uLCAxNSBTZXB0ZW1iZXIgMjAyNicsCiAgICAgICAgICAgIGFzc2VydGlvbj0nMTIgY2FwdHVyZSBzZXNzaW9ucyByZXByZXNlbnQgMTIgZGlmZmVyZW50IHBoeXNpY2FsIHR5cmVzJyxpbmRlcGVuZGVudGx5X3ZlcmlmaWVkPUZhbHNlKSwKICAgICAgICBncm91cHM9Z3JvdXBzLHJvd3M9cm93cyxwb2ludF9uYW1lcz1QT0lOVFMsCiAgICAgICAgbW9kZWw9J2hybmV0X3cxOC5tc19hdWdfaW4xaycsZmVhdHVyZV9sb2NhdGlvbj0nJyxmZWF0dXJlX2luZGljZXM9WzEsMiwzLDRdLAogICAgICAgIHByZXRyYWluZWRfcmVwbz0ndGltbS9ocm5ldF93MTgubXNfYXVnX2luMWsnLHByZXRyYWluZWRfcmV2aXNpb249JzdlMmM1NTgzNzY5ZjU0NTE0ZmQ4N2UzYmE5ZGU0MDhlMzNlYWJhMGYnLAogICAgICAgIHBhY2thZ2VzPXsndGltbSc6JzEuMC4xNSd9LGlucHV0X2h3PVs1MTIsMzg0XSxiYXRjaF9zaXplPTIsZXBvY2hzPTYwLHNlZWRzPVsxLDIsM10sCiAgICAgICAgb3B0aW1pemVyPSdBZGFtVycsbHI9MC4wMDAxLHdlaWdodF9kZWNheT0wLjAxLGxyX3NjaGVkdWxlPSdjb3NpbmVfZXBvY2gnLAogICAgICAgIGF1Z21lbnRhdGlvbj0nbm9uZTsgZGV0ZXJtaW5pc3RpYyBpZGVudGl0eSB0cmFuc2Zvcm0nLGJhdGNobm9ybT0nZnJvemVuIHJ1bm5pbmcgc3RhdGlzdGljcycsCiAgICAgICAgbG9zcz0nc2l4IGhvcml6b250YWwgR2F1c3NpYW4gdGFyZ2V0cywgc2lnbWE9MS41IGZlYXR1cmUgcGl4ZWxzOyBtZWFuIGNyb3NzIGVudHJvcHknLAogICAgICAgIGVuZHBvaW50PSdmaXhlZCBlcG9jaCA2MDsgbm8gYmVzdC10ZXN0IG9yIGJlc3QtdmFsaWRhdGlvbiBjaGVja3BvaW50IHNlbGVjdGlvbicsCiAgICAgICAgdmlzaWJpbGl0eT0nYWxsIGxhYmVscyB2aXNpYmxlOyBubyB2aXNpYmlsaXR5IGNsYXNzaWZpZXIgb3IgcmVqZWN0aW9uIHZhbGlkYXRpb24nLAogICAgICAgIHFhPSdtZWNoYW5pY2FsIGNoZWNrcyBhbmQgcmV2aWV3IGRpYWdyYW1zOyBodW1hbiBhY2N1cmFjeSBub3QgY2VydGlmaWVkJywKICAgICAgICBzY3JpcHRzPXtuOmZpbGVfc2hhKFBhdGgoX19maWxlX18pLnBhcmVudC9uKSBmb3IgbiBpbiBbJ2hybmV0X3Byb3RvY29sLnB5JywnaHJuZXRfcnVudGltZS5weScsJ3M5X2V4cGFuc2lvbi5weScsJ3M5X3BpbG90LnB5J119LAogICAgICAgIGxpbWl0YXRpb25zPVsnb25seSAxMiB1c2VyLWNvbmZpcm1lZCB0eXJlczsgc21hbGwgdHdvLXR5cmUgdmFsaWRhdGlvbi90ZXN0IHNldHMnLAogICAgICAgICAgICAnb2xkIHBpbG90IGltYWdlcyB3ZXJlIGRldmVsb3BtZW50IGV2aWRlbmNlOyBzYW1lIHR5cmUgaWRlbnRpdGllcyB3ZXJlIHNlZW4gaW4gcGlsb3QgYW5hbHlzaXMnLAogICAgICAgICAgICAnbm90IHVudG91Y2hlZCBleHRlcm5hbC1jb2hvcnQgdmFsaWRhdGlvbjsgbm8gcGh5c2ljYWwgYW5nbGVzIG9yIGhlYWx0aHktcmVmZXJlbmNlIGluZmVyZW5jZScsCiAgICAgICAgICAgICdubyBtYXRjaGVkIHNlZ21lbnRhdGlvbiBjb21wYXJpc29uIHVudGlsIGJhc2VsaW5lIHRyYWluLXNldCBvdmVybGFwIGlzIHJlc29sdmVkJ10pCiAgICByZXR1cm4gY2ZnCmRlZiBwcmVmaXgoY2ZnKTpyZXR1cm4gZidzOS97VkVSU0lPTn0ve3NoYShjYW5vbmljYWwoY2ZnKSl9JwpkZWYgcm9vdF9kYXRhKHJvb3Q9JycpOgogICAgaWYgcm9vdDpyZXR1cm4gUGF0aChyb290KQogICAgZm91bmQ9bGlzdChQYXRoKCcva2FnZ2xlL2lucHV0JykuZ2xvYignKiovbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpKQogICAgaWYgbGVuKGZvdW5kKSE9MTpyYWlzZSBWYWx1ZUVycm9yKCdBdHRhY2ggT05FIFRpcmUgRGF0YXNldCBQcmVwYXJlZCBkYXRhc2V0LCBvciBzZXQgREFUQV9ST09UIHRvIEZJTkFMJykKICAgIHJldHVybiBmb3VuZFswXS5wYXJlbnQucGFyZW50CmRlZiB2YWxpZGF0ZV9pbWFnZXMoY2ZnLHJvb3QpOgogICAgaGFzaGVzPXtrOnNldCgpIGZvciBrIGluIGNmZ1snZ3JvdXBzJ119CiAgICBmb3IgciBpbiBjZmdbJ3Jvd3MnXToKICAgICAgICBwYXRoPShQYXRoKHJvb3QpL3JbJ29yaWdpbmFsX3JlbGF0aXZlX3BhdGgnXSkucmVzb2x2ZSgpCiAgICAgICAgYXNzZXJ0IHBhdGguaXNfcmVsYXRpdmVfdG8oUGF0aChyb290KS5yZXNvbHZlKCkpCiAgICAgICAgYXNzZXJ0IGZpbGVfc2hhKHBhdGgpPT1yWydpbWFnZV9zaGEyNTYnXSxyWydwaWxvdF9pZCddKycgb3JpZ2luYWwgZGlmZmVycycKICAgICAgICB3aXRoIEltYWdlLm9wZW4ocGF0aCkgYXMgaW06YXNzZXJ0IGltLnNpemU9PShyWyd3aWR0aCddLHJbJ2hlaWdodCddKQogICAgICAgIGhhc2hlc1tyWydyb2xlJ11dLmFkZChyWydpbWFnZV9zaGEyNTYnXSkKICAgIGFzc2VydCBub3QgaGFzaGVzWyd0cmFpbiddJmhhc2hlc1sndmFsaWRhdGlvbiddIGFuZCBub3QgaGFzaGVzWyd0cmFpbiddJmhhc2hlc1sndGVzdCddIGFuZCBub3QgaGFzaGVzWyd0ZXN0J10maGFzaGVzWyd2YWxpZGF0aW9uJ10KZGVmIHB1Ymxpc2goZm9sZGVyLHBhdGgsdG9rZW4pOgogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpCiAgICBmb2xkZXI9UGF0aChmb2xkZXIpCiAgICAjIE9uZSBzeW5jaHJvbm91cyB3cml0ZXI7IG5vIGNoZWNrcG9pbnQgY2hhbmdlcyBkdXJpbmcgdGhpcyBhdG9taWMgcmVwbyBjb21taXQuCiAgICByZXN1bHQ9cmV0cnkobGFtYmRhOkhmQXBpKHRva2VuPXRva2VuKS51cGxvYWRfZm9sZGVyKHJlcG9faWQ9UkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLAogICAgICAgIGZvbGRlcl9wYXRoPXN0cihmb2xkZXIpLHBhdGhfaW5fcmVwbz1wYXRoLGlnbm9yZV9wYXR0ZXJucz1bJyoudG1wJywnKi5sb2NrJ10sCiAgICAgICAgY29tbWl0X21lc3NhZ2U9J0hSTmV0IGdlb21ldHJ5IGR1cmFibGUgc25hcHNob3QnKSkKICAgIHByaW50KCdIRiBwdWJsaWNhdGlvbiBzdWNjZWVkZWQ6JyxyZXN1bHQub2lkLGZsdXNoPVRydWUpO3JldHVybiByZXN1bHQub2lkCmRlZiBwcmVmbGlnaHQod29yayxyb290LHRva2VuLHVwbG9hZD1UcnVlKToKICAgIGNmZz1jb250cmFjdCh3b3JrKTt2YWxpZGF0ZV9pbWFnZXMoY2ZnLHJvb3QpCiAgICBmb2xkZXI9UGF0aCh3b3JrKS8ncHJlZmxpZ2h0Jy9zaGEoY2Fub25pY2FsKGNmZykpO2ZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIHdyaXRlKGZvbGRlci8nQ09OVFJBQ1QuanNvbicsY2ZnKQogICAgY291bnRzPXtrOnN1bShyWydyb2xlJ109PWsgZm9yIHIgaW4gY2ZnWydyb3dzJ10pIGZvciBrIGluIGNmZ1snZ3JvdXBzJ119CiAgICBmbGFncz1bXQogICAgZm9yIHIgaW4gY2ZnWydyb3dzJ106CiAgICAgICAgIyBPcmlnaW5hbC1pbWFnZSBwb2ludCBvdmVybGF5czsgbm90IHN5bnRoZXRpYyBsYWJlbHMuIEV2ZXJ5IGltYWdlIGlzIGluY2x1ZGVkLgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihQYXRoKHJvb3QpL3JbJ29yaWdpbmFsX3JlbGF0aXZlX3BhdGgnXSkgYXMgaW06CiAgICAgICAgICAgIGltPWltLmNvbnZlcnQoJ1JHQicpO2RyYXc9SW1hZ2VEcmF3LkRyYXcoaW0pCiAgICAgICAgICAgIGZvciBqLHggaW4gZW51bWVyYXRlKHJbJ3gnXSk6CiAgICAgICAgICAgICAgICB4eD14KihyWyd3aWR0aCddLTEpO3k9clsnZ3VpZGVfeSddW2ovLzJdCiAgICAgICAgICAgICAgICBkcmF3LmVsbGlwc2UoKHh4LTgseS04LHh4KzgseSs4KSxvdXRsaW5lPSdjeWFuJyx3aWR0aD0zKQogICAgICAgICAgICAgICAgaWYgbWluKHh4LHJbJ3dpZHRoJ10tMS14eCk8PTEwOmZsYWdzLmFwcGVuZChkaWN0KHBpbG90X2lkPXJbJ3BpbG90X2lkJ10scG9pbnQ9UE9JTlRTW2pdLHJlYXNvbj0nbmVhciBlZGdlOyBkaWFnbm9zdGljIG9ubHknKSkKICAgICAgICAgICAgaW0udGh1bWJuYWlsKCg1NzYsNzY4KSk7aW0uc2F2ZShmb2xkZXIvKHJbJ3BpbG90X2lkJ10rJy5qcGcnKSxxdWFsaXR5PTg4KQogICAgd3JpdGUoZm9sZGVyLydRQS5qc29uJyxkaWN0KGNvdW50cz1jb3VudHMsZmxhZ3M9ZmxhZ3MsYWxsX3Zpc2libGVfcG9pbnRzPTcyMCwKICAgICAgICBsYWJlbF9xdWFsaXR5PSdub3QgaW5kZXBlbmRlbnRseSBjZXJ0aWZpZWQnLGlkZW50aXR5PWNmZ1snaWRlbnRpdHknXSkpCiAgICB3cml0ZShmb2xkZXIvJ1NUQVRVUy5qc29uJyxkaWN0KHN0YXR1cz0ncHJlZmxpZ2h0X3Bhc3NlZCcscHJvdG9jb2w9c2hhKGNhbm9uaWNhbChjZmcpKSwKICAgICAgICBuZXh0PSdSdW4gTkIyNyBzbW9rZSB0ZXN0OyBkbyBub3QgaW50ZXJwcmV0IGZvcm1hdCBjaGVja3MgYXMgcGh5c2ljYWwgdmFsaWRhdGlvbicpKQogICAgZm9yIG4gaW4gY2ZnWydzY3JpcHRzJ106c2h1dGlsLmNvcHkyKFBhdGgoX19maWxlX18pLnBhcmVudC9uLGZvbGRlci9uKQogICAgaWYgdXBsb2FkOnB1Ymxpc2goZm9sZGVyLHByZWZpeChjZmcpKycvcHJlZmxpZ2h0Jyx0b2tlbikKICAgIHByaW50KCdMb2NrZWQgaW1hZ2UgY291bnRzOicsY291bnRzLCc7IHR5cmVzOicse2s6bGVuKHYpIGZvciBrLHYgaW4gY2ZnWydncm91cHMnXS5pdGVtcygpfSxmbHVzaD1UcnVlKQogICAgcmV0dXJuIGZvbGRlcgo='))
(WORK/'hrnet_runtime.py').write_bytes(base64.b64decode('...'))
(WORK/'s9_expansion.py').write_bytes(base64.b64decode('IiIiQm91bmRlZCBTOSBnZW9tZXRyeSBhbm5vdGF0aW9uIGV4cGFuc2lvbjsgbm8gdHJhaW5pbmcgb3IgaW5mZXJyZWQgaGVhbHRoIGxhYmVscy4iIiIKaW1wb3J0IGJhc2U2NAppbXBvcnQgY3N2CmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgemlwZmlsZQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKaW1wb3J0IHM5X3BpbG90IGFzIHAKClZFUlNJT049J3M5LWdlb21ldHJ5LWxhYmVscy1yMi1hbGwxMjAnClRBUkdFVD0xMjAKTUFYX1BBQ0tBR0VfQllURVM9MTYwKjEwMjQqKjIKZGVmIHNlbGVjdGlvbihyb3dzLCBwaWxvdCk6CiAgICBleGNsdWRlZD17clsnaW1hZ2VfaWQnXSBmb3IgciBpbiBwaWxvdFsnaW1hZ2VzJ119CiAgICBwb29scz17czpzb3J0ZWQoW3IgZm9yIHIgaW4gcm93cyBpZiByWydzZXNzaW9uX2dyb3VwJ109PXMgYW5kIHJbJ2ltYWdlX2lkJ10gbm90IGluIGV4Y2x1ZGVkXSwKICAgICAgICAgICAgICAgICAgIGtleT1sYW1iZGEgcjpwLmRpZ2VzdChyWydpbWFnZV9pZCddLmVuY29kZSgpKSkgZm9yIHMgaW4gc29ydGVkKHtyWydzZXNzaW9uX2dyb3VwJ10gZm9yIHIgaW4gcm93c30pfQogICAgY2hvc2VuPVtdCiAgICB3aGlsZSBsZW4oY2hvc2VuKTxUQVJHRVQ6CiAgICAgICAgYmVmb3JlPWxlbihjaG9zZW4pCiAgICAgICAgZm9yIHMscG9vbCBpbiBwb29scy5pdGVtcygpOgogICAgICAgICAgICBpZiBwb29sIGFuZCBsZW4oY2hvc2VuKTxUQVJHRVQ6IGNob3Nlbi5hcHBlbmQocG9vbC5wb3AoMCkpCiAgICAgICAgaWYgbGVuKGNob3Nlbik9PWJlZm9yZTpyYWlzZSBWYWx1ZUVycm9yKCdJbnN1ZmZpY2llbnQgdW5pcXVlIG5ldyBpbWFnZXMnKQogICAgYXNzZXJ0IGxlbih7clsnaW1hZ2VfaWQnXSBmb3IgciBpbiBjaG9zZW59KT09VEFSR0VUCiAgICByZXR1cm4gY2hvc2VuCgpkZWYgcHJlcGFyZShyb290LCBwaWxvdCwgdGVtcGxhdGUsIG91dCk6CiAgICByb290PVBhdGgocm9vdCk7bWFuaWZlc3RfcGF0aD1yb290LydtYW5pZmVzdHMvY2xlYW5fbWFuaWZlc3QuY3N2JwogICAgcmF3PW1hbmlmZXN0X3BhdGgucmVhZF9ieXRlcygpCiAgICBhc3NlcnQgcC5kaWdlc3QocmF3KT09cGlsb3RbJ21hbmlmZXN0X3NoYTI1NiddLCAnUHJlcGFyZWQgZGF0YXNldCBtYW5pZmVzdCBkaWZmZXJzIGZyb20gdGhlIHBpbG90JwogICAgcm93cz1saXN0KGNzdi5EaWN0UmVhZGVyKHJhdy5kZWNvZGUoJ3V0Zi04LXNpZycpLnNwbGl0bGluZXMoKSkpCiAgICBhc3NlcnQgbGVuKHJvd3MpPT00MTggYW5kIGxlbih7clsnaW1hZ2VfaWQnXSBmb3IgciBpbiByb3dzfSk9PTQxOAogICAgc2VsZWN0ZWQ9c2VsZWN0aW9uKHJvd3MscGlsb3QpCiAgICBhbGxvY2F0aW9uPVtkaWN0KGltYWdlX2lkPXJbJ2ltYWdlX2lkJ10sc2Vzc2lvbj1yWydzZXNzaW9uX2dyb3VwJ10sCiAgICAgICAgICAgICAgICAgICAgIGZpbGVfc2hhMjU2PXJbJ2ZpbGVfc2hhMjU2J10pIGZvciBpLHIgaW4gZW51bWVyYXRlKHNlbGVjdGVkKV0KICAgIHBsYW49ZGljdCh2ZXJzaW9uPVZFUlNJT04sdGFyZ2V0X2ltYWdlcz0xMjAsYmF0Y2hfc2l6ZT0xMjAsYWxsb2NhdGlvbj1hbGxvY2F0aW9uLAogICAgICAgICAgICAgIHBpbG90X3BhY2thZ2U9cGlsb3RbJ3BhY2thZ2VfaWQnXSxzcGxpdF9zdGF0dXM9J1VOTE9DS0VEX3JlcXVpcmVzX3BoeXNpY2FsX3R5cmVfaWRlbnRpdHlfcmV2aWV3JywKICAgICAgICAgICAgICBwdXJwb3NlPSdzaXggaW1hZ2UtcmVsYXRpdmUgdHJlYWQtdHJhbnNpdGlvbiBwb2ludHM7IG5vdCBwaHlzaWNhbCBhbGlnbm1lbnQnLAogICAgICAgICAgICAgIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLGhlYWx0aHlfcmVmZXJlbmNlX2VsaWdpYmxlPUZhbHNlKQogICAgcGxhbl9oYXNoPXAuZGlnZXN0KHAuY2Fub25pY2FsKHBsYW4pKQogICAgcmVjb3Jkcz1bXTtlbWJlZGRlZD1bXQogICAgZm9yIGksciBpbiBlbnVtZXJhdGUoc2VsZWN0ZWQpOgogICAgICAgIHBhdGg9KHJvb3QvclsncmVsYXRpdmVfcGF0aCddKS5yZXNvbHZlKCkKICAgICAgICBhc3NlcnQgcGF0aC5pc19yZWxhdGl2ZV90byhyb290LnJlc29sdmUoKSkKICAgICAgICBpbWFnZV9yYXc9cGF0aC5yZWFkX2J5dGVzKCk7YXNzZXJ0IHAuZGlnZXN0KGltYWdlX3Jhdyk9PXJbJ2ZpbGVfc2hhMjU2J10KICAgICAgICB3aXRoIEltYWdlLm9wZW4ocGF0aCkgYXMgaW06CiAgICAgICAgICAgIGFzc2VydCBpbS5mb3JtYXQ9PSdKUEVHJzt3LGg9aW0uc2l6ZQogICAgICAgIGFzc2VydCAodyxoKT09KGludChyWyd3aWR0aCddKSxpbnQoclsnaGVpZ2h0J10pKQogICAgICAgIHJlYz1kaWN0KHBpbG90X2lkPWYnR3tpKzE6MDNkfScsaW1hZ2VfaWQ9clsnaW1hZ2VfaWQnXSwKICAgICAgICAgICAgICAgICBpbWFnZV9zaGEyNTY9clsnZmlsZV9zaGEyNTYnXSx3aWR0aD13LGhlaWdodD1oLGd1aWRlX3k9W3JvdW5kKChoLTEpKmYpIGZvciBmIGluICguMjUsLjUsLjc1KV0sCiAgICAgICAgICAgICAgICAgc2Vzc2lvbj1yWydzZXNzaW9uX2dyb3VwJ10sZm9sZD1pbnQoclsnZm9sZF9pZCddKSxvcmlnaW5hbF9yZWxhdGl2ZV9wYXRoPXJbJ3JlbGF0aXZlX3BhdGgnXSkKICAgICAgICByZWNvcmRzLmFwcGVuZChyZWMpO2VtYmVkZGVkLmFwcGVuZChyZWN8eydpbWFnZSc6ZidpbWFnZXMvR3tpKzE6MDNkfS5qcGcnfSkKICAgIHByb3RvY29sPWRpY3QodmVyc2lvbj1wLlZFUlNJT04sZXhwYW5zaW9uX3ZlcnNpb249VkVSU0lPTixwbGFuX2hhc2g9cGxhbl9oYXNoLAogICAgICAgIHBvaW50cz1wLlBPSU5UUyxpbWFnZXM9cmVjb3Jkcyxjb3VudD0xMjAsdHJhaW5pbmdfYXBwcm92ZWQ9RmFsc2UsCiAgICAgICAgc291cmNlX3NoYTI1Nj1wLmRpZ2VzdChQYXRoKF9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLHRlbXBsYXRlX3NoYTI1Nj1wLmRpZ2VzdChQYXRoKHRlbXBsYXRlKS5yZWFkX2J5dGVzKCkpKQogICAgcGFja2FnZT1wLmRpZ2VzdChwLmNhbm9uaWNhbChwcm90b2NvbCkpO3Byb3RvY29sWydwYWNrYWdlX2lkJ109cGFja2FnZQogICAgZGVzdD1QYXRoKG91dCkvcGFja2FnZTtkZXN0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgaHRtbD1QYXRoKHRlbXBsYXRlKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykKICAgICMgRXh0ZXJuYWwgbmF0aXZlIEpQRUdzIGtlZXAgdGhlIEhUTUwgc21hbGw7IG9ubHkgdGhlIGN1cnJlbnQgaW1hZ2UgaXMgZGVjb2RlZC4KICAgIGh0bWw9aHRtbC5yZXBsYWNlKCcxMiBpbWFnZXMuIE9uZSBzbWFsbCBwaWxvdC4gTm8gdHJhaW5pbmcgeWV0LicsJ0FsbCAxMjAgaW1hZ2VzIOKAlCBvbmUgYW5ub3RhdGlvbiBwcm9qZWN0LicpCiAgICBodG1sPWh0bWwucmVwbGFjZSgnVGhpcyBpcyBhIGZlYXNpYmlsaXR5IGNoZWNrLCBub3QgYSA0MTgtaW1hZ2UgYXNzaWdubWVudC4nLAogICAgICAgICdUaGlzIGlzIG9uZSAxMjAtaW1hZ2UgZ2VvbWV0cnkgYXNzaWdubWVudC4gWW91IGNhbiBhbm5vdGF0ZSBhbGwgaW1hZ2VzIHdpdGhvdXQgYmF0Y2ggc3dpdGNoaW5nIG9yIGludGVybWVkaWF0ZSBhcHByb3ZhbC4gU2F2ZSBKU09OIHRvIHdvcmsgYWNyb3NzIHNpdHRpbmdzLicpCiAgICBodG1sPWh0bWwucmVwbGFjZSgnb3IgYXR0YWNoIHRoYXQgSlNPTiB0byBLYWdnbGUgYW5kIHJ1biBOQjIyJywnb3IgYXR0YWNoIHRoYXQgSlNPTiB0byBLYWdnbGUgYW5kIHJ1biBOQjI1JykKICAgIGh0bWw9aHRtbC5yZXBsYWNlKCcxMi8xMicsJzEyMC8xMjAnKS5yZXBsYWNlKCJuKycvMTIgcmV2aWV3ZWQnIiwibisnLycrREFUQS5pbWFnZXMubGVuZ3RoKycgcmV2aWV3ZWQnIikKICAgIGh0bWw9aHRtbC5yZXBsYWNlKCdNYXRoLm1pbigxMSxpbmRleCsxKScsJ01hdGgubWluKERBVEEuaW1hZ2VzLmxlbmd0aC0xLGluZGV4KzEpJykKICAgIGh0bWw9aHRtbC5yZXBsYWNlKCd2LmFubm90YXRpb25zLmxlbmd0aD4xMicsJ3YuYW5ub3RhdGlvbnMubGVuZ3RoPkRBVEEuaW1hZ2VzLmxlbmd0aCcpCiAgICBodG1sPWh0bWwucmVwbGFjZSgnc3RvcCBhZnRlciAxMiBpbWFnZXMuJywnYWxsIDEyMCBpbWFnZXMgYXJlIGF2YWlsYWJsZTsgc2F2ZSBhZ2FpbiB3aGVuZXZlciB5b3UgcGF1c2UuJykKICAgIGh0bWw9aHRtbC5yZXBsYWNlKCdTdG9wIHRoZXJlOyB3YWl0IGZvciBmZWVkYmFjayBiZWZvcmUgYW55IGxhcmdlciBiYXRjaCBvciBtb2RlbCB0cmFpbmluZy4nLAogICAgICAgICdSZXR1cm4geW91ciBzYXZlZCBKU09OIHdoZW4gcmVhZHkuIFRyYWluaW5nIGZvbGxvd3MgZmluYWwgbGFiZWwgYW5kIHNwbGl0IHJldmlldzsgbm8gaW50ZXJtZWRpYXRlIDEyLWltYWdlIHBhdXNlLicpCiAgICBodG1sPWh0bWwucmVwbGFjZSgnUzkg4oCUIDEyLWltYWdlIGd1aWRlZCBwaWxvdCcsJ1M5IOKAlCBhbGwgMTIwIGdlb21ldHJ5IGltYWdlcycpCiAgICBodG1sPWh0bWwucmVwbGFjZSgnPGgzPldvcmtlZCBleGFtcGxlcycsCiAgICAgICAgJzxwIGNsYXNzPSJub3RpY2UiPjxiPlF1YWxpdHkgcnVsZTo8L2I+IEFuIGltYWdlIGVkZ2UgaXMgbm90IGEgdmlzaWJsZSBib3VuZGFyeS4gU2VsZWN0IE91dHNpZGUgZnJhbWUgd2hlbiBjbGlwcGVkLiBJZiB5b3UgY2Fubm90IGRpc3Rpbmd1aXNoIHRyZWFkIGZyb20gc2hvdWxkZXIsIHNlbGVjdCBVbmNlcnRhaW47IGRvIG5vdCB1c2UgdGhlIHNpbGhvdWV0dGUgaW5zdGVhZC4gVmlzaWJsZSBpc3N1ZSByZXF1aXJlcyBhIHNob3J0IG5vdGUuIENoZWNrIGFsbCBzaXggcG9pbnQgc3RhdGVzIGJlZm9yZSBzYXZpbmcuIFRoZXNlIGFyZSBnZW9tZXRyeSBsYWJlbHMsIG5vdCBkZWZlY3QgbWFza3Mgb3IgaGVhbHRoIGxhYmVscy48L3A+PGgzPldvcmtlZCBleGFtcGxlcycpCiAgICBwYXlsb2FkPWRpY3QodmVyc2lvbj1wLlZFUlNJT04scGFja2FnZV9pZD1wYWNrYWdlLHBvaW50cz1wLlBPSU5UUyxpbWFnZXM9W3trOnYgZm9yIGssdiBpbiByLml0ZW1zKCkgaWYgayBub3QgaW4gKCdzZXNzaW9uJywnZm9sZCcsJ2ltYWdlX2lkJywnb3JpZ2luYWxfcmVsYXRpdmVfcGF0aCcpfSBmb3IgciBpbiBlbWJlZGRlZF0pCiAgICBodG1sPWh0bWwucmVwbGFjZSgnX19QSUxPVF9EQVRBX18nLGpzb24uZHVtcHMocGF5bG9hZCkucmVwbGFjZSgnPCcsJ1xcdTAwM2MnKSkKICAgIGFzc2VydCBsZW4oaHRtbC5lbmNvZGUoKSk8PXAuTUFYX0JZVEVTLCAnUGFja2FnZSB0b28gbGFyZ2U6IHN0b3BwZWQgd2l0aG91dCBkZWdyYWRpbmcgb3JpZ2luYWxzJwogICAgKGRlc3QvJ0FOTk9UQVRFLmh0bWwnKS53cml0ZV90ZXh0KGh0bWwsZW5jb2Rpbmc9J3V0Zi04JykKICAgIGltYWdlcz1kZXN0LydpbWFnZXMnO2ltYWdlcy5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZm9yIHJlZiBpbiByZWNvcmRzOgogICAgICAgIHJhd19pbWFnZT0ocm9vdC9yZWZbJ29yaWdpbmFsX3JlbGF0aXZlX3BhdGgnXSkucmVhZF9ieXRlcygpCiAgICAgICAgYXNzZXJ0IHAuZGlnZXN0KHJhd19pbWFnZSk9PXJlZlsnaW1hZ2Vfc2hhMjU2J10KICAgICAgICAoaW1hZ2VzLyhyZWZbJ3BpbG90X2lkJ10rJy5qcGcnKSkud3JpdGVfYnl0ZXMocmF3X2ltYWdlKQogICAgcC53cml0ZV9qc29uKGRlc3QvJ01BTklGRVNULmpzb24nLHByb3RvY29sKTtwLndyaXRlX2pzb24oZGVzdC8nUExBTi5qc29uJyxwbGFuKQogICAgIyBTYW1lIGJsYW5rIGlkZW50aXR5IGxlZGdlciBpbiBlYWNoIGJhdGNoLCB1c2VyIHN1cHBsaWVzIGZhY3R1YWwgaWRlbnRpdHkgb25seS4KICAgIHAud3JpdGVfanNvbihkZXN0LydTRVNTSU9OX0lERU5USVRZLmpzb24nLGRpY3QoaW5zdHJ1Y3Rpb249J1NhbWUgcGh5c2ljYWwgdHlyZSBtdXN0IHVzZSBzYW1lIGFub255bW91cyBJRCBhY3Jvc3Mgc2Vzc2lvbnMuIExlYXZlIHVua25vd24gaWYgdW5zdXJlLiBEbyBub3QgaW52ZW50IGlkZW50aXRpZXMuJywKICAgICAgICBzZXNzaW9ucz1bZGljdChzZXNzaW9uPXMscGh5c2ljYWxfdHlyZV9pZD1Ob25lLHJlY29yZF9ub3RlPScnKSBmb3IgcyBpbiBzb3J0ZWQoe3JbJ3Nlc3Npb25fZ3JvdXAnXSBmb3IgciBpbiByb3dzfSldKSkKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGRlc3QvJ0FMTF8xMjBfSU1BR0VTLnppcCcsJ3cnLHppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyB6OgogICAgICAgIGZvciBuYW1lIGluIFsnQU5OT1RBVEUuaHRtbCcsJ01BTklGRVNULmpzb24nLCdTRVNTSU9OX0lERU5USVRZLmpzb24nXToKICAgICAgICAgICAgei53cml0ZShkZXN0L25hbWUsYXJjbmFtZT1uYW1lKQogICAgICAgIGZvciBwYXRoIGluIHNvcnRlZChpbWFnZXMuZ2xvYignKi5qcGcnKSk6ei53cml0ZShwYXRoLGFyY25hbWU9J2ltYWdlcy8nK3BhdGgubmFtZSkKICAgIGFzc2VydCAoZGVzdC8nQUxMXzEyMF9JTUFHRVMuemlwJykuc3RhdCgpLnN0X3NpemU8PU1BWF9QQUNLQUdFX0JZVEVTCiAgICBwLndyaXRlX2pzb24oZGVzdC8nU1RBVFVTLmpzb24nLGRpY3Qoc3RhdHVzPSdhd2FpdGluZ19hbm5vdGF0aW9ucycsaW1hZ2VfY291bnQ9MTIwLHBsYW5faGFzaD1wbGFuX2hhc2gsCiAgICAgICAgcGFja2FnZV9pZD1wYWNrYWdlLHppcF9ieXRlcz0oZGVzdC8nQUxMXzEyMF9JTUFHRVMuemlwJykuc3RhdCgpLnN0X3NpemUsCiAgICAgICAgdHJhaW5pbmdfYXBwcm92ZWQ9RmFsc2UsZnVsbF9zOV9jb21wbGV0ZT1GYWxzZSkpCiAgICByZXR1cm4gZGVzdAoKZGVmIGludGFrZShhbm5vdGF0aW9uLCBtYW5pZmVzdCwgb3V0KToKICAgIGFubm90YXRpb249UGF0aChhbm5vdGF0aW9uKTtyYXc9YW5ub3RhdGlvbi5yZWFkX2J5dGVzKCkKICAgIGFzc2VydCBsZW4ocmF3KTw9MTAyNCoqMgogICAgdmFsdWU9anNvbi5sb2FkcyhyYXcpCiAgICBhc3NlcnQgbWFuaWZlc3QuZ2V0KCdleHBhbnNpb25fdmVyc2lvbicpPT1WRVJTSU9OCiAgICBhc3NlcnQgcC5kaWdlc3QocC5jYW5vbmljYWwoe2s6diBmb3Igayx2IGluIG1hbmlmZXN0Lml0ZW1zKCkgaWYgayE9J3BhY2thZ2VfaWQnfSkpPT1tYW5pZmVzdFsncGFja2FnZV9pZCddCiAgICByZWNvcmRzPXZhbGlkYXRlKHZhbHVlLG1hbmlmZXN0KQogICAgZG9uZT1zdW0oclsnY29tcGxldGUnXSBmb3IgciBpbiByZWNvcmRzKQogICAgZGVzdD1QYXRoKG91dCkvcC5kaWdlc3QocmF3KTtkZXN0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgKGRlc3QvJ0FOTk9UQVRJT05TLmpzb24nKS53cml0ZV9ieXRlcyhyYXcpCiAgICBwLndyaXRlX2pzb24oZGVzdC8nUkVWSUVXLmpzb24nLGRpY3Qoc3RhdHVzPSduZWVkc19odW1hbl9yZXZpZXcnIGlmIGRvbmU9PTEyMCBlbHNlICdwYXJ0aWFsX2Fubm90YXRpb24nLAogICAgICAgIGNvbXBsZXRlX2ltYWdlcz1kb25lLGV4cGVjdGVkX2ltYWdlcz0xMjAscmVjb3Jkcz1yZWNvcmRzLHBhY2thZ2VfaWQ9bWFuaWZlc3RbJ3BhY2thZ2VfaWQnXSwKICAgICAgICBwbGFuX2hhc2g9bWFuaWZlc3RbJ3BsYW5faGFzaCddLGFubm90YXRpb25zX3NoYTI1Nj1wLmRpZ2VzdChyYXcpLHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlKSkKICAgIHAud3JpdGVfanNvbihkZXN0LydTVEFUVVMuanNvbicsZGljdChzdGF0dXM9J21lY2hhbmljYWxfaW50YWtlX29ubHknLGZ1bGxfczlfY29tcGxldGU9RmFsc2UsdHJhaW5pbmdfYXBwcm92ZWQ9RmFsc2UpKQogICAgcHJpbnQoZid7ZG9uZX0vMTIwIGNvbXBsZXRlLiBQcm9ncmVzcyBzYXZlZDsgZmluYWwgaHVtYW4gcmV2aWV3IGFuZCBzcGxpdCBsb2NrIHN0aWxsIHJlcXVpcmVkLicpCiAgICByZXR1cm4gZGVzdAoKZGVmIHZhbGlkYXRlKHZhbHVlLG1hbmlmZXN0KToKICAgIGxhYmVscz12YWx1ZS5nZXQoJ2Fubm90YXRpb25zJykKICAgIGlmIG5vdCBpc2luc3RhbmNlKGxhYmVscyxsaXN0KSBvciBsZW4obGFiZWxzKT4xMjA6cmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgYXQgbW9zdCAxMjAgcmVjb3JkcycpCiAgICBpZHM9W2EuZ2V0KCdwaWxvdF9pZCcpIGZvciBhIGluIGxhYmVscyBpZiBpc2luc3RhbmNlKGEsZGljdCldCiAgICBpZiBsZW4oaWRzKSE9bGVuKGxhYmVscykgb3IgbGVuKHNldChpZHMpKSE9bGVuKGlkcyk6cmFpc2UgVmFsdWVFcnJvcignSW52YWxpZC9kdXBsaWNhdGUgaW1hZ2UgcmVjb3JkcycpCiAgICAjIFJldXNlIHN0cmljdCBjb29yZGluYXRlL3NjaGVtYSB2YWxpZGF0aW9uIHdpdGhvdXQgYWx0ZXJpbmcgb2xkIE5CMjEvTkIyMiBiZWhhdmlvdXIuCiAgICByZXN1bHQ9W10KICAgIGZvciBpIGluIHJhbmdlKDAsbWF4KDEsbGVuKGxhYmVscykpLDEyKToKICAgICAgICByZXN1bHQuZXh0ZW5kKHAudmFsaWRhdGVfYW5ub3RhdGlvbnModmFsdWV8eydhbm5vdGF0aW9ucyc6bGFiZWxzW2k6aSsxMl19LG1hbmlmZXN0KSkKICAgIHJldHVybiByZXN1bHQKCmRlZiBwdWJsaXNoKGZvbGRlciwgdG9rZW4sIHByZWZpeCk6CiAgICBpbXBvcnQgdGltZQogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpCiAgICBhbGxvd2VkPVsnQUxMXzEyMF9JTUFHRVMuemlwJywnTUFOSUZFU1QuanNvbicsJ1BMQU4uanNvbicsJ1NFU1NJT05fSURFTlRJVFkuanNvbicsJ1NUQVRVUy5qc29uJywKICAgICAgICAgICAgICdBTk5PVEFUSU9OUy5qc29uJywnUkVWSUVXLmpzb24nLCdzOV9leHBhbnNpb24ucHknXQogICAgYXNzZXJ0IHN1bShmLnN0YXQoKS5zdF9zaXplIGZvciBmIGluIFBhdGgoZm9sZGVyKS5pdGVyZGlyKCkgaWYgZi5uYW1lIGluIGFsbG93ZWQpPD1NQVhfUEFDS0FHRV9CWVRFUysyKjEwMjQqKjIKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcj1IZkFwaSh0b2tlbj10b2tlbikudXBsb2FkX2ZvbGRlcihyZXBvX2lkPXAuUkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLGZvbGRlcl9wYXRoPXN0cihmb2xkZXIpLAogICAgICAgICAgICAgICAgcGF0aF9pbl9yZXBvPXByZWZpeCxhbGxvd19wYXR0ZXJucz1hbGxvd2VkLGNvbW1pdF9tZXNzYWdlPSdTOSBib3VuZGVkIGdlb21ldHJ5IGFubm90YXRpb24gYmF0Y2g7IG5vIHRyYWluaW5nJykKICAgICAgICAgICAgcHJpbnQoJ0hGIHB1YmxpY2F0aW9uIHN1Y2NlZWRlZDonLHIub2lkKTtyZXR1cm4gci5vaWQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHN0YXR1cz1nZXRhdHRyKGdldGF0dHIoZSwncmVzcG9uc2UnLE5vbmUpLCdzdGF0dXNfY29kZScsTm9uZSkKICAgICAgICAgICAgaWYgc3RhdHVzIG5vdCBpbiAoTm9uZSw0MjksNTAwLDUwMiw1MDMsNTA0KSBvciBhdHRlbXB0PT00OnJhaXNlCiAgICAgICAgICAgIGhpbnQ9Z2V0YXR0cihnZXRhdHRyKGUsJ3Jlc3BvbnNlJyxOb25lKSwnaGVhZGVycycse30pLmdldCgnUmV0cnktQWZ0ZXInLCcwJykKICAgICAgICAgICAgd2FpdD1tYXgoMTAqMioqYXR0ZW1wdCxmbG9hdChoaW50KSBpZiBoaW50LmlzZGlnaXQoKSBlbHNlIDApCiAgICAgICAgICAgIHByaW50KGYnVXBsb2FkIGJhY2tvZmYge3dhaXR9czsgZmlsZXMgc2FmZSBsb2NhbGx5LicpO3RpbWUuc2xlZXAod2FpdCkK'))
(WORK/'s9_pilot.py').write_bytes(base64.b64decode('IiIiU21hbGwsIGxvc3NsZXNzIFM5IGFubm90YXRpb24gZmVhc2liaWxpdHkgcGlsb3QuIE5vIG1vZGVsIHRyYWluaW5nIG9yIGhlYWx0aCBpbmZlcmVuY2UuIiIiCmltcG9ydCBiYXNlNjQKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKClZFUlNJT04gPSAnczktaW5wdXQtcGlsb3QtcjEnClJFUE8gPSAnU2hhbm11azQ2MjIvdHlyZS13ZWFyLXN0dWR5JwpNQVhfQllURVMgPSAyMCAqIDEwMjQgKiAxMDI0ClBPSU5UUyA9IFsnbGVmdF91cHBlcicsICdyaWdodF91cHBlcicsICdsZWZ0X21pZGRsZScsICdyaWdodF9taWRkbGUnLCAnbGVmdF9sb3dlcicsICdyaWdodF9sb3dlciddClNUQVRFUyA9IFsndmlzaWJsZScsICdvY2NsdWRlZCcsICdvdXRzaWRlX2ZyYW1lJywgJ3VuY2VydGFpbiddClRSSUFHRSA9IFsnbm9fdmlzaWJsZV9pc3N1ZScsICd2aXNpYmxlX2lzc3VlJywgJ3VuYXNzZXNzYWJsZSddCgpkZWYgZGlnZXN0KHJhdyk6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKQoKZGVmIGNhbm9uaWNhbCh2YWx1ZSk6CiAgICByZXR1cm4ganNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCcsJywgJzonKSwgZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoKQoKZGVmIHdyaXRlX2pzb24ocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAnLnRtcCcpCiAgICB0bXAud3JpdGVfYnl0ZXMoY2Fub25pY2FsKHZhbHVlKSk7IHRtcC5yZXBsYWNlKHBhdGgpCgpkZWYgZGlzY292ZXIocm9vdD0nL2thZ2dsZS9pbnB1dCcpOgogICAgY2FuZGlkYXRlcyA9IHNvcnRlZChQYXRoKHJvb3QpLmdsb2IoJyoqL21hbmlmZXN0cy9jbGVhbl9tYW5pZmVzdC5jc3YnKSkKICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0F0dGFjaCBPTkUgVGlyZSBEYXRhc2V0IFByZXBhcmVkIHBhY2thZ2UsIG9yIHNldCBEQVRBX1JPT1QgdG8gaXRzIEZJTkFMIGRpcmVjdG9yeS4nKQogICAgcmV0dXJuIGNhbmRpZGF0ZXNbMF0ucGFyZW50LnBhcmVudAoKZGVmIHByZXBhcmUoZGF0YV9yb290LCBvdXRwdXQsIHRlbXBsYXRlKToKICAgIGRhdGFfcm9vdCwgb3V0cHV0ID0gUGF0aChkYXRhX3Jvb3QpLCBQYXRoKG91dHB1dCkKICAgIHdpdGggKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLm9wZW4oZW5jb2Rpbmc9J3V0Zi04LXNpZycsIG5ld2xpbmU9JycpIGFzIGY6CiAgICAgICAgYWxsX3Jvd3MgPSBsaXN0KGNzdi5EaWN0UmVhZGVyKGYpKQogICAgaWYgbGVuKGFsbF9yb3dzKSAhPSA0MTggb3IgbGVuKHtyWydpbWFnZV9pZCddIGZvciByIGluIGFsbF9yb3dzfSkgIT0gNDE4OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHRoZSBmcm96ZW4gNDE4IHVuaXF1ZSBjbGVhbiBvcmlnaW5hbHMuJykKICAgIGdyb3VwcyA9IHNvcnRlZCh7clsnc2Vzc2lvbl9ncm91cCddIGZvciByIGluIGFsbF9yb3dzfSkKICAgIGlmIGxlbihncm91cHMpICE9IDEyOiByYWlzZSBWYWx1ZUVycm9yKCdFeHBlY3RlZCAxMiBjYXB0dXJlIHNlc3Npb25zLCBub3QgYSBuZXcgZGF0YXNldC4nKQogICAgY2hvc2VuID0gW10KICAgIGZvciBncm91cCBpbiBncm91cHM6CiAgICAgICAgcnIgPSBzb3J0ZWQoW3IgZm9yIHIgaW4gYWxsX3Jvd3MgaWYgclsnc2Vzc2lvbl9ncm91cCddID09IGdyb3VwXSwga2V5PWxhbWJkYSByOiByWydpbWFnZV9pZCddKQogICAgICAgIGNob3Nlbi5hcHBlbmQocnJbbGVuKHJyKS8vMl0pICAjIEZpeGVkIG1lZGlhbi1JRCByZXByZXNlbnRhdGl2ZSwgbm90IHNlbGVjdGVkIGJ5IG1vZGVsIHBlcmZvcm1hbmNlLgogICAgcmVjb3JkcywgZW1iZWRkZWQgPSBbXSwgW10KICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShjaG9zZW4pOgogICAgICAgIHAgPSAoZGF0YV9yb290L3JbJ3JlbGF0aXZlX3BhdGgnXSkucmVzb2x2ZSgpCiAgICAgICAgaWYgbm90IHAuaXNfcmVsYXRpdmVfdG8oZGF0YV9yb290LnJlc29sdmUoKSk6IHJhaXNlIFZhbHVlRXJyb3IoJ0ltYWdlIHBhdGggZXNjYXBlcyBkYXRhIHJvb3QnKQogICAgICAgIHJhdyA9IHAucmVhZF9ieXRlcygpCiAgICAgICAgaWYgZGlnZXN0KHJhdykgIT0gclsnZmlsZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcihmJ0ltYWdlIGhhc2ggbWlzbWF0Y2g6IHBpbG90IHtpKzF9JykKICAgICAgICB3aXRoIEltYWdlLm9wZW4ocCkgYXMgaW06CiAgICAgICAgICAgIGlmIGltLmZvcm1hdCAhPSAnSlBFRycgb3IgaW0uc2l6ZSAhPSAoaW50KHJbJ3dpZHRoJ10pLCBpbnQoclsnaGVpZ2h0J10pKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHVuY2hhbmdlZCBuYXRpdmUgSlBFRyBkaW1lbnNpb25zJykKICAgICAgICAgICAgd2lkdGgsIGhlaWdodCA9IGltLnNpemUKICAgICAgICBpdGVtID0gZGljdChwaWxvdF9pZD1mJ1B7aSsxOjAyZH0nLCBpbWFnZV9pZD1yWydpbWFnZV9pZCddLCBzZXNzaW9uPXJbJ3Nlc3Npb25fZ3JvdXAnXSwKICAgICAgICAgICAgICAgICAgICBmb2xkPWludChyWydmb2xkX2lkJ10pLCBvcmlnaW5hbF9yZWxhdGl2ZV9wYXRoPXJbJ3JlbGF0aXZlX3BhdGgnXSwKICAgICAgICAgICAgICAgICAgICBpbWFnZV9zaGEyNTY9ZGlnZXN0KHJhdyksIHdpZHRoPXdpZHRoLCBoZWlnaHQ9aGVpZ2h0LAogICAgICAgICAgICAgICAgICAgIGd1aWRlX3k9W3JvdW5kKChoZWlnaHQtMSkqZikgZm9yIGYgaW4gKC4yNSwuNSwuNzUpXSwKICAgICAgICAgICAgICAgICAgICB0cmFuc2Zvcm09J2lkZW50aXR5OyBuYXRpdmUgcGl4ZWxzOyBubyBjcm9wLCByZXNpemUgb3IgcmVjb21wcmVzc2lvbicpCiAgICAgICAgcmVjb3Jkcy5hcHBlbmQoaXRlbSkKICAgICAgICBlbWJlZGRlZC5hcHBlbmQoe2s6aXRlbVtrXSBmb3IgayBpbiBbJ3BpbG90X2lkJywnaW1hZ2Vfc2hhMjU2Jywnd2lkdGgnLCdoZWlnaHQnLCdndWlkZV95J119CiAgICAgICAgICAgICAgICAgICAgICAgIHwgeydpbWFnZSc6J2RhdGE6aW1hZ2UvanBlZztiYXNlNjQsJytiYXNlNjQuYjY0ZW5jb2RlKHJhdykuZGVjb2RlKCl9KQogICAgdGVtcGxhdGUgPSBQYXRoKHRlbXBsYXRlKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykKICAgIHByb3RvY29sID0gZGljdCh2ZXJzaW9uPVZFUlNJT04sIG1hbmlmZXN0X3NoYTI1Nj1kaWdlc3QoKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLnJlYWRfYnl0ZXMoKSksCiAgICAgICAgICAgICAgICAgICAgc291cmNlX3NoYTI1Nj1kaWdlc3QoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKSwgdGVtcGxhdGVfc2hhMjU2PWRpZ2VzdCh0ZW1wbGF0ZS5lbmNvZGUoKSksCiAgICAgICAgICAgICAgICAgICAgcG9pbnRzPVBPSU5UUywgaW1hZ2VzPXJlY29yZHMsIGNvdW50PTEyLAogICAgICAgICAgICAgICAgICAgIHB1cnBvc2U9J0ZlYXNpYmlsaXR5IG9ubHk7IHByb3Bvc2VkIGltYWdlLXBsYW5lIHRyZWFkLWJvdW5kYXJ5IGxhYmVsczsgbm8gdHJhaW5pbmcgYXBwcm92YWwnLAogICAgICAgICAgICAgICAgICAgIGhlYWx0aHlfcmVmZXJlbmNlX2VsaWdpYmxlPUZhbHNlLCBodW1hbl9yZXZpZXdfcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhY2thZ2VfaWQgPSBkaWdlc3QoY2Fub25pY2FsKHByb3RvY29sKSkKICAgIHByb3RvY29sWydwYWNrYWdlX2lkJ10gPSBwYWNrYWdlX2lkCiAgICBkZXN0ID0gb3V0cHV0L3BhY2thZ2VfaWQ7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF5bG9hZCA9IGRpY3QocGFja2FnZV9pZD1wYWNrYWdlX2lkLCB2ZXJzaW9uPVZFUlNJT04sIHBvaW50cz1QT0lOVFMsIGltYWdlcz1lbWJlZGRlZCkKICAgIHBhZ2UgPSB0ZW1wbGF0ZS5yZXBsYWNlKCdfX1BJTE9UX0RBVEFfXycsIGpzb24uZHVtcHMocGF5bG9hZCkucmVwbGFjZSgnPCcsJ1xcdTAwM2MnKSkKICAgIGlmIGxlbihwYWdlLmVuY29kZSgpKSA+IE1BWF9CWVRFUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdMb3NzbGVzcyAxMi1pbWFnZSBwYWdlIGV4Y2VlZHMgMjAgTWlCOiBzdG9wcGVkIHdpdGhvdXQgcmVkdWNpbmcgZGV0YWlsLiBSZXF1ZXN0IGEgc21hbGxlciBiYXRjaC4nKQogICAgKGRlc3QvJ0FOTk9UQVRFLmh0bWwnKS53cml0ZV90ZXh0KHBhZ2UsIGVuY29kaW5nPSd1dGYtOCcpCiAgICB3cml0ZV9qc29uKGRlc3QvJ01BTklGRVNULmpzb24nLCBwcm90b2NvbCkKICAgIHppcF9wYXRoID0gZGVzdC8nUElMT1RfMTJfSU1BR0VTLnppcCcKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHppcF9wYXRoLCAndycsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBuYW1lIGluIFsnQU5OT1RBVEUuaHRtbCcsJ01BTklGRVNULmpzb24nXToKICAgICAgICAgICAgaW5mbz16aXBmaWxlLlppcEluZm8obmFtZSxkYXRlX3RpbWU9KDIwMjYsOSwxNCwwLDAsMCkpCiAgICAgICAgICAgIGluZm8uY29tcHJlc3NfdHlwZT16aXBmaWxlLlpJUF9ERUZMQVRFRAogICAgICAgICAgICBhcmNoaXZlLndyaXRlc3RyKGluZm8sKGRlc3QvbmFtZSkucmVhZF9ieXRlcygpKQogICAgaWYgemlwX3BhdGguc3RhdCgpLnN0X3NpemUgPiBNQVhfQllURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ1BhY2thZ2UgZXhjZWVkcyAyMCBNaUI7IG5vIHVwbG9hZCBhbGxvd2VkJykKICAgIHdyaXRlX2pzb24oZGVzdC8nU1RBVFVTLmpzb24nLCBkaWN0KHZlcnNpb249VkVSU0lPTiwgcGFja2FnZV9pZD1wYWNrYWdlX2lkLAogICAgICAgIHN0YXR1cz0nYXdhaXRpbmdfcGlsb3RfYW5ub3RhdGlvbicsIGltYWdlX2NvdW50PTEyLCB6aXBfYnl0ZXM9emlwX3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgaHRtbF9ieXRlcz0oZGVzdC8nQU5OT1RBVEUuaHRtbCcpLnN0YXQoKS5zdF9zaXplLCB6aXBfc2hhMjU2PWRpZ2VzdCh6aXBfcGF0aC5yZWFkX2J5dGVzKCkpLAogICAgICAgIGZ1bGxfczlfY29tcGxldGU9RmFsc2UsIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSkpCiAgICBwcmludChmJzEyIG5hdGl2ZSBpbWFnZXM7IFpJUCB7emlwX3BhdGguc3RhdCgpLnN0X3NpemUvMioqMjA6LjJmfSBNaUIuIE9wZW4gQU5OT1RBVEUuaHRtbCBhZnRlciBleHRyYWN0aW9uLicpCiAgICByZXR1cm4gZGVzdAoKZGVmIHZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLCBtYW5pZmVzdCk6CiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCkgb3IgdmFsdWUuZ2V0KCdwYWNrYWdlX2lkJykgIT0gbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHBhY2thZ2UgaWRlbnRpdHkgbWlzbWF0Y2guIEltcG9ydCB0aGUgSlNPTiBpbnRvIHRoZSBtYXRjaGluZyBwYWdlLicpCiAgICBpZiB2YWx1ZS5nZXQoJ3ZlcnNpb24nKSAhPSBWRVJTSU9OOiByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHNjaGVtYSBtaXNtYXRjaCcpCiAgICBsYWJlbHMgPSB2YWx1ZS5nZXQoJ2Fubm90YXRpb25zJykKICAgIGlmIG5vdCBpc2luc3RhbmNlKGxhYmVscywgbGlzdCkgb3IgbGVuKGxhYmVscykgPiAxMjogcmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgYXQgbW9zdCAxMiBhbm5vdGF0aW9uIHJlY29yZHMnKQogICAgYnlfaWQgPSB7clsncGlsb3RfaWQnXTpyIGZvciByIGluIG1hbmlmZXN0WydpbWFnZXMnXX07IHNlZW49c2V0KCk7IGNoZWNrZWQ9W10KICAgIGZvciBhIGluIGxhYmVsczoKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShhLGRpY3QpOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGFubm90YXRpb24gZW50cnknKQogICAgICAgIHBpZD1hLmdldCgncGlsb3RfaWQnKQogICAgICAgIGlmIHBpZCBub3QgaW4gYnlfaWQgb3IgcGlkIGluIHNlZW46IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gb3IgZHVwbGljYXRlZCBwaWxvdCBJRCcpCiAgICAgICAgc2Vlbi5hZGQocGlkKTsgcmVmPWJ5X2lkW3BpZF0KICAgICAgICBpZiBhLmdldCgnaW1hZ2Vfc2hhMjU2JykgIT0gcmVmWydpbWFnZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignSW1hZ2UgaWRlbnRpdHkgbWlzbWF0Y2gnKQogICAgICAgIHBvaW50cz1hLmdldCgncG9pbnRzJyx7fSkKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwb2ludHMsZGljdCkgb3Igc2V0KHBvaW50cyktc2V0KFBPSU5UUyk6IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gbGFuZG1hcmsgbmFtZXMnKQogICAgICAgIGNvbXBsZXRlPVRydWU7IHZpc2libGU9MAogICAgICAgIGZvciBqLG5hbWUgaW4gZW51bWVyYXRlKFBPSU5UUyk6CiAgICAgICAgICAgIHA9cG9pbnRzLmdldChuYW1lKQogICAgICAgICAgICBpZiBwIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlOyBjb250aW51ZQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwLGRpY3QpIG9yIHAuZ2V0KCdzdGF0ZScpIG5vdCBpbiBTVEFURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzaWJpbGl0eSBzdGF0ZScpCiAgICAgICAgICAgIGlmIHBbJ3N0YXRlJ109PSd2aXNpYmxlJzoKICAgICAgICAgICAgICAgIHgseT1wLmdldCgneCcpLHAuZ2V0KCd5JykKICAgICAgICAgICAgICAgIGlmIGFueSh0eXBlKHYpIG5vdCBpbiAoZmxvYXQsaW50KSBvciBub3QgbWF0aC5pc2Zpbml0ZSh2KSBmb3IgdiBpbiAoeCx5KSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVmlzaWJsZSBwb2ludHMgbmVlZCBmaW5pdGUgY29vcmRpbmF0ZXMnKQogICAgICAgICAgICAgICAgaWYgbm90ICgwPD14PHJlZlsnd2lkdGgnXSBhbmQgMDw9eTxyZWZbJ2hlaWdodCddKTogcmFpc2UgVmFsdWVFcnJvcignUG9pbnQgb3V0IG9mIGJvdW5kcycpCiAgICAgICAgICAgICAgICBpZiB5ICE9IHJlZlsnZ3VpZGVfeSddW2ovLzJdOiByYWlzZSBWYWx1ZUVycm9yKCdQb2ludCBtdXN0IGJlIG9uIGl0cyBleGFjdCBob3Jpem9udGFsIGd1aWRlJykKICAgICAgICAgICAgICAgIHZpc2libGUrPTEKICAgICAgICAgICAgZWxpZiBwLmdldCgneCcpIGlzIG5vdCBOb25lIG9yIHAuZ2V0KCd5JykgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZpc2libGUgcG9pbnQgbXVzdCBub3QgaGF2ZSBndWVzc2VkIGNvb3JkaW5hdGVzJykKICAgICAgICBmb3IgbGV2ZWwgaW4gWyd1cHBlcicsJ21pZGRsZScsJ2xvd2VyJ106CiAgICAgICAgICAgIGwscj1wb2ludHMuZ2V0KCdsZWZ0XycrbGV2ZWwse30pLHBvaW50cy5nZXQoJ3JpZ2h0XycrbGV2ZWwse30pCiAgICAgICAgICAgIGlmIGwuZ2V0KCdzdGF0ZScpPT1yLmdldCgnc3RhdGUnKT09J3Zpc2libGUnIGFuZCBsWyd4J10+PXJbJ3gnXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0xlZnQvcmlnaHQgcG9pbnRzIGFyZSBjcm9zc2VkIG9yIGVxdWFsJykKICAgICAgICB0cmlhZ2U9YS5nZXQoJ3Zpc3VhbF9yZXZpZXcnKQogICAgICAgIGlmIHRyaWFnZSBpcyBub3QgTm9uZSBhbmQgdHJpYWdlIG5vdCBpbiBUUklBR0U6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzdWFsIHJldmlldycpCiAgICAgICAgaWYgdHJpYWdlIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlCiAgICAgICAgbm90ZT1hLmdldCgnbm90ZScsJycpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uobm90ZSxzdHIpIG9yIGxlbihub3RlKT4yMDAwOiByYWlzZSBWYWx1ZUVycm9yKCdOb3RlIGV4Y2VlZHMgMjAwMCBjaGFyYWN0ZXJzJykKICAgICAgICBpZiB0cmlhZ2U9PSd2aXNpYmxlX2lzc3VlJyBhbmQgbm90IG5vdGUuc3RyaXAoKTogY29tcGxldGU9RmFsc2UKICAgICAgICBldmlkZW5jZT1hLmdldCgnaW5kZXBlbmRlbnRfcmVjb3JkJywndW5rbm93bicpCiAgICAgICAgaWYgZXZpZGVuY2Ugbm90IGluIFsndW5rbm93bicsJ2F2YWlsYWJsZSddOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGV2aWRlbmNlIGF2YWlsYWJpbGl0eScpCiAgICAgICAgIyBBIHVzZXIgYXNzZXJ0aW9uIGlzIHJlY29yZGVkLCBORVZFUiBhdXRvbWF0aWNhbGx5IHByb21vdGVkIHRvIHZlcmlmaWVkIGhlYWx0aC4KICAgICAgICBjaGVja2VkLmFwcGVuZChkaWN0KHBpbG90X2lkPXBpZCwgaW1hZ2VfaWQ9cmVmWydpbWFnZV9pZCddLCBjb21wbGV0ZT1jb21wbGV0ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZpc2libGVfcG9pbnRzPXZpc2libGUsIHZpc3VhbF9yZXZpZXc9dHJpYWdlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGVwZW5kZW50X3JlY29yZD1ldmlkZW5jZSwgbm90ZT1ub3RlKSkKICAgIHJldHVybiBjaGVja2VkCgpkZWYgcmV2aWV3KGFubm90YXRpb25fcGF0aCwgcGFja2FnZSwgb3V0cHV0KToKICAgIHBhY2thZ2UsIG91dHB1dD1QYXRoKHBhY2thZ2UpLFBhdGgob3V0cHV0KQogICAgcmF3PVBhdGgoYW5ub3RhdGlvbl9wYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGxlbihyYXcpPjEwMjQqMTAyNDogcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIHRoZSBhbm5vdGF0aW9uLW9ubHkgSlNPTiAodW5kZXIgMSBNaUIpLCBub3QgaW1hZ2VzIG9yIFpJUCcpCiAgICB2YWx1ZT1qc29uLmxvYWRzKHJhdyk7IG1hbmlmZXN0PWpzb24ubG9hZHMoKHBhY2thZ2UvJ01BTklGRVNULmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGlmIGRpZ2VzdChjYW5vbmljYWwoe2s6diBmb3Igayx2IGluIG1hbmlmZXN0Lml0ZW1zKCkgaWYgayE9J3BhY2thZ2VfaWQnfSkpIT1tYW5pZmVzdC5nZXQoJ3BhY2thZ2VfaWQnKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNYW5pZmVzdCBjb250ZW50IGhhc2ggbWlzbWF0Y2gnKQogICAgcmVzdWx0PXZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLG1hbmlmZXN0KQogICAgZGVzdD1vdXRwdXQvbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXS8ncmV2aWV3cycvZGlnZXN0KHJhdyk7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAoZGVzdC8nQU5OT1RBVElPTlMuanNvbicpLndyaXRlX2J5dGVzKHJhdykKICAgIGRvbmU9c3VtKHJbJ2NvbXBsZXRlJ10gZm9yIHIgaW4gcmVzdWx0KQogICAgd3JpdGVfanNvbihkZXN0LydSRVZJRVcuanNvbicsZGljdChzdGF0dXM9J25lZWRzX2h1bWFuX3JldmlldycgaWYgZG9uZT09MTIgZWxzZSAncGFydGlhbF9hbm5vdGF0aW9uJywKICAgICAgIGNvbXBsZXRlX2ltYWdlcz1kb25lLCBleHBlY3RlZF9pbWFnZXM9MTIsIHJlY29yZHM9cmVzdWx0LCBwYWNrYWdlX2lkPW1hbmlmZXN0WydwYWNrYWdlX2lkJ10sCiAgICAgICBhbm5vdGF0aW9uc19zaGEyNTY9ZGlnZXN0KHJhdyksIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgIG5leHRfYWN0aW9uPSdTZW5kIHRoZSBKU09OIGFuZCByZXZpZXcgcmVwb3J0IHRvIHRoZSBhc3Npc3RhbnQuIERvIG5vdCBsYWJlbCBtb3JlIG9yIHRyYWluIHlldC4nKSkKICAgIHByaW50KGYnUGlsb3QgY29tcGxldGlvbjoge2RvbmV9LzEyLiBNZWNoYW5pY2FsIGNoZWNrcyBvbmx5OyBodW1hbiByZXZpZXcgc3RpbGwgcmVxdWlyZWQuIE5vIHRyYWluaW5nIHN0YXJ0ZWQuJykKICAgIHJldHVybiBkZXN0CgpkZWYgcHVibGlzaChmb2xkZXIsIHRva2VuLCBwcmVmaXgpOgogICAgIiIiT25lIGNvbW1pdCBwZXIgbWFqb3IgYWN0aW9uOyBzbWFsbCBpbW11dGFibGUgY29udGVudC4gTm8gY2xhaW0vaGVhcnRiZWF0IHdyaXRlcy4iIiIKICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgZm9sZGVyPVBhdGgoZm9sZGVyKQogICAgYWxsb3dlZD1bJ1NUQVRVUy5qc29uJywnTUFOSUZFU1QuanNvbicsJ1BJTE9UXzEyX0lNQUdFUy56aXAnLCdSRVZJRVcuanNvbicsJ0FOTk9UQVRJT05TLmpzb24nLAogICAgICAgICAgICAgJ3M5X3BpbG90LnB5JywncGlsb3RfdGVtcGxhdGUuaHRtbCddCiAgICBmaWxlcz1bcCBmb3IgcCBpbiBmb2xkZXIuaXRlcmRpcigpIGlmIHAubmFtZSBpbiBhbGxvd2VkIGFuZCBwLmlzX2ZpbGUoKV0KICAgIGlmIHN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGZpbGVzKT5NQVhfQllURVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIGV4Y2VlZHMgMjAgTWlCIGxpbWl0JykKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzdWx0PUhmQXBpKHRva2VuPXRva2VuKS51cGxvYWRfZm9sZGVyKHJlcG9faWQ9UkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLGZvbGRlcl9wYXRoPXN0cihmb2xkZXIpLAogICAgICAgICAgICAgICAgcGF0aF9pbl9yZXBvPXByZWZpeCxhbGxvd19wYXR0ZXJucz1hbGxvd2VkLGNvbW1pdF9tZXNzYWdlPSdTOSBzbWFsbCBhbm5vdGF0aW9uIHBpbG90OyBubyBtb2RlbCB0cmFpbmluZycpCiAgICAgICAgICAgIHByaW50KCdIRiBwdWJsaWNhdGlvbiBzdWNjZWVkZWQ6JyxyZXN1bHQub2lkKTsgcmV0dXJuIHJlc3VsdC5vaWQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgcmVzcG9uc2U9Z2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKTsgc3RhdHVzPWdldGF0dHIocmVzcG9uc2UsJ3N0YXR1c19jb2RlJyxOb25lKQogICAgICAgICAgICBpZiBzdGF0dXMgbm90IGluICg0MjksNTAwLDUwMiw1MDMsNTA0KSBvciBhdHRlbXB0PT00OgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdIRiBwdWJsaWNhdGlvbiBkaWQgbm90IGNvbXBsZXRlLiBLZWVwIGxvY2FsIG91dHB1dHMgYW5kIHJldHJ5IHRoaXMgY2VsbDsgbm8gdHJhaW5pbmcgd2FzIHJ1bi4nKSBmcm9tIE5vbmUKICAgICAgICAgICAgaGludD1nZXRhdHRyKHJlc3BvbnNlLCdoZWFkZXJzJyx7fSkuZ2V0KCdSZXRyeS1BZnRlcicsJycpCiAgICAgICAgICAgIHdhaXQ9bWF4KDEwKjIqKmF0dGVtcHQsZmxvYXQoaGludCkgaWYgc3RyKGhpbnQpLmlzZGlnaXQoKSBlbHNlIDApCiAgICAgICAgICAgIHByaW50KGYnSEYgYmFja29mZjoge3dhaXQ6LjBmfXMuIFNhdmVkIGxvY2FsIG91dHB1dHMgcmVtYWluIGF2YWlsYWJsZS4nKQogICAgICAgICAgICB1bnRpbD10aW1lLm1vbm90b25pYygpK3dhaXQKICAgICAgICAgICAgd2hpbGUgdGltZS5tb25vdG9uaWMoKTx1bnRpbDogdGltZS5zbGVlcChtaW4oNSxtYXgoMCx1bnRpbC10aW1lLm1vbm90b25pYygpKSkpCg=='))
sys.path.insert(0,str(WORK))
import importlib, hrnet_protocol as hp
hp=importlib.reload(hp)
from kaggle_secrets import UserSecretsClient
TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
if not TOKEN: raise RuntimeError('Enable HF_TOKEN in Kaggle secrets')
from IPython.display import FileLink, display


In [2]:
# No dataset attachment required for reporting. Reads only small status/metric files.
from huggingface_hub import HfApi,hf_hub_download
cfg=hp.contract(WORK);protocol=hp.sha(hp.canonical(cfg));prefix=hp.prefix(cfg)
rev=HfApi(token=TOKEN).repo_info(hp.REPO,repo_type='dataset').sha
OUT=WORK/'report'/protocol;OUT.mkdir(parents=True,exist_ok=True)
def fetch(path):
    f=hp.retry(lambda:hf_hub_download(hp.REPO,path,repo_type='dataset',revision=rev,token=TOKEN))
    if Path(f).stat().st_size>10*1024**2:raise ValueError('Unexpectedly large report artifact')
    return hp.read_json(f)
train=[r for r in cfg['rows'] if r['role']=='train'];test={r['pilot_id']:r for r in cfg['rows'] if r['role']=='test'}
mean=[sum(r['x'][i] for r in train)/len(train) for i in range(6)]
baseline=sum(abs(mean[i]-r['x'][i]) for r in test.values() for i in range(6))/(6*len(test))
results=[]
for seed in cfg['seeds']:
    remote=prefix+f'/runs/seed{seed}'
    st=fetch(remote+'/STATUS.json')
    assert st['status']=='completed' and st['completed_epochs']==60 and st['protocol']==protocol and st['seed']==seed
    metric=fetch(remote+'/TEST_FINAL.json');rows=metric['records']
    assert len(rows)==len(test)*6 and {(r['pilot_id'],r['point']) for r in rows}=={(pid,n) for pid in test for n in hp.POINTS}
    errors=[]
    for r in rows:
        target=test[r['pilot_id']]['x'][hp.POINTS.index(r['point'])]
        assert r['label']==target and r['tyre']==test[r['pilot_id']]['session']
        error=abs(r['prediction']-target);assert abs(error-r['error_width_fraction'])<1e-9
        errors.append(error)
    score=sum(errors)/len(errors);assert abs(score-metric['mean_width_error'])<1e-9
    repair=fetch(remote+'/RUNTIME_REPAIR.json')
    results.append(dict(seed=seed,mean_width_error=score,train_mean_baseline=baseline,
        runtime_repair_revision=repair['revision'],runtime_repair_sha256=repair['source_sha256'],
        per_tyre={t:sum(abs(r['prediction']-r['label']) for r in rows if r['tyre']==t)/sum(r['tyre']==t for r in rows) for t in cfg['groups']['test']}))
report=dict(source_revision=rev,protocol=protocol,results=results,
    claim='HRNet versus train-only constant-coordinate baseline; NOT a matched SegFormer comparison',
    full_s9_complete=False,limitations=cfg['limitations'])
hp.write(OUT/'REPORT.json',report);hp.write(OUT/'CONTRACT.json',cfg)
hp.publish(OUT,prefix+'/report',TOKEN)
display(FileLink(str(OUT/'REPORT.json')));print(json.dumps(report,indent=2))


STATUS.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

TEST_FINAL.json: 0.00B [00:00, ?B/s]

RUNTIME_REPAIR.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

STATUS.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

TEST_FINAL.json: 0.00B [00:00, ?B/s]

...

TEST_FINAL.json: 0.00B [00:00, ?B/s]

RUNTIME_REPAIR.json: 0.00B [00:00, ?B/s]

HF publication succeeded: a92c0f9c5c1b78c6a06e13d51e18722195230658


/kaggle/working/hrnet_geometry/report/351c6638783996f46cf9efd99ec6689de726e045288054f240193cea61385712/REPORT.json

{
  "source_revision": "96166fb19f9bf63d1acb2bd2ee8b55b0ffdacadd",
  "protocol": "351c6638783996f46cf9efd99ec6689de726e045288054f240193cea61385712",
  "results": [
    {
      "seed": 1,
      "mean_width_error": 0.01390767721937562,
      "train_mean_baseline": 0.03554712220184273,
      "runtime_repair_revision": "amp-same-batch-retry-2026-09-15-r1",
      "runtime_repair_sha256": "2bce03798d6b531ba4c1f8d91599fe565e47ddfd0f674027c41db17c1b395afa",
      "per_tyre": {
        "mileage_100000_plus__session_006": 0.005057249212111959,
        "new_tire__session_001": 0.02275810522663928
      }
    },
    {
      "seed": 2,
      "mean_width_error": 0.012835461712814182,
      "train_mean_baseline": 0.03554712220184273,
      "runtime_repair_revision": "amp-same-batch-retry-2026-09-15-r1",
      "runtime_repair_sha256": "2bce03798d6b531ba4c1f8d91599fe565e47ddfd0f674027c41db17c1b395afa",
      "per_tyre": {
        "mileage_100000_plus__session_006": 0.005423934714159678,
        "new_ti